# Semantic Scholar API Demo

This notebook demonstrates how to use the `SemanticScholarClient` for academic impact metrics enrichment.

## Overview

Semantic Scholar provides:
- Citation counts and influential citations
- Abstracts and TL;DR summaries
- Author information with affiliations
- Fields of study and topic classifications
- Paper recommendations
- Open access PDF locations

## Installation

```bash
pip install pyeuropepmc
```

## API Key

While not required, you can get a free API key from [Semantic Scholar API](https://www.semanticscholar.org/product/api) for higher rate limits.

### Using .env file

Create a `.env` file in the project root with your API key:

```
SEMANTIC_SCHOLAR_API_KEY=your_api_key_here
```

The client will automatically load this key and use Bearer authentication.


In [19]:
# Load environment variables from .env file
import os
from dotenv import load_dotenv

# Load .env file from project root (parent of current directory)
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == '10-semantic-scholar' else current_dir
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)
print(f"Loaded .env from: {env_path}")

# Check if SEMANTIC_SCHOLAR_API_KEY is set
api_key = os.getenv('SEMANTIC_SCHOLAR_API_KEY')
if api_key:
    print(f"Semantic Scholar API key found: {api_key[:5]}... (hidden)")
    print("Using authenticated endpoint (100k requests/month)")
else:
    print("No API key configured - using free tier (25k requests/month)")
    print("Get a free key at: https://www.semanticscholar.org/product/api")


Loaded .env from: /home/jhe24/AID-PAIS/pyEuropePMC_project/examples/.env
Semantic Scholar API key found: mfhyq... (hidden)
Using authenticated endpoint (100k requests/month)


In [20]:
# Initialize the Semantic Scholar client
from pyeuropepmc.enrichment import SemanticScholarClient

# Initialize client (api_key will be loaded from .env if available)
client = SemanticScholarClient(
    rate_limit_delay=1.0,  # Delay between requests in seconds
    cache_config=None       # Disable caching for fresh data
)

# Verify client initialization
auth_header = client.session.headers.get('Authorization', '')
if auth_header.startswith('Bearer '):
    print("Semantic Scholar API key configured (using Authorization: Bearer header)")
    print(f"API key present: {client.api_key is not None}")
else:
    print("Semantic Scholar client initialized (no API key configured)")

print("\nUsing professional Semantic Scholar client (danielnsilva/semanticscholar v0.12.0)")
print("Features: typed responses, async support, automatic retries, full API coverage")


Semantic Scholar API key configured (using Authorization: Bearer header)
API key present: True

Using professional Semantic Scholar client (danielnsilva/semanticscholar v0.12.0)
Features: typed responses, async support, automatic retries, full API coverage


## API Key Configuration Options

The client supports two ways to configure your API key:

### Option 1: Using Environment Variable (Recommended)

Set your API key in a `.env` file or environment variable. The client automatically uses it:

```python
import os
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = "your_api_key_here"
client = SemanticScholarClient()
```

### Option 2: Passing API Key Directly

Pass the key directly to the constructor:

```python
client = SemanticScholarClient(api_key="your_api_key_here")
```

The API key enables:
- **100k requests/month** (vs 25k on free tier)
- **Bearer authentication** via `Authorization: Bearer` header
- **Faster rate limits** (4x more requests)

To get a free API key, visit: https://www.semanticscholar.org/product/api


## Professional Semantic Scholar Library

PyEuropePMC now uses the `danielnsilva/semanticscholar` professional library (v0.12.0) for enhanced API integration.

### Key Features:
- ✅ **Typed objects**: `Paper`, `Author`, `Venue` with attribute access
- ✅ **Automatic retries**: Uses `tenacity` for resilient API calls
- ✅ **Full API coverage**: Graph API, Recommendations API, Datasets API
- ✅ **Async support**: Async client available for concurrent requests
- ✅ **Built-in pagination**: Automatic pagination handling
- ✅ **Production-ready**: 461 stars on GitHub

### Benefits:
- More robust error handling
- Better rate limit management
- Type safety throughout the response
- Simpler code with less boilerplate

The professional library is initialized automatically when you create a `SemanticScholarClient`. All existing functionality is preserved while gaining the benefits of the professional client.

### Note on Authentication:

The `danielnsilva/semanticscholar` library has a known bug where it uses `{'x-api-key': api_key}` header instead of the correct `{'Authorization': 'Bearer <token>'}` format. **PyEuropePMC automatically patches this internally**, so you don't need to do anything - your API key will work correctly regardless of the underlying library bug.

## Basic Paper Enrichment

Enrich a paper using DOI or Semantic Scholar ID.

In [21]:
# Example DOI
DOI = "10.1371/journal.pone.0308090"

# Enrich paper using DOI
result = client.enrich(identifier=DOI)

if result:
    print(f"Title: {result.get('title')}")
    print(f"Year: {result.get('year')}")
    print(f"Citation count: {result.get('citation_count')}")
    print(f"Influential citation count: {result.get('influential_citation_count')}")
    print(f"Authors: {len(result.get('authors', []))}")
    print(f"Fields of study: {result.get('fields_of_study')}")
    print(f"Abstract: {result.get('abstract')[:200]}..." if result.get('abstract') else "No abstract")

Title: Associations of dietary pattern, insulin resistance and risk of developing metabolic syndrome among Chinese population
Year: 2024
Citation count: 3
Influential citation count: 0
Authors: 11
Fields of study: ['Medicine']
Abstract: Evidence regarding the role of dietary patterns in metabolic syndrome (MetS) is limited. The mechanistic links between dietary patterns, insulin resistance, and MetS are not fully understood. This stu...


## Enrich Using Semantic Scholar ID

In [22]:
# First, get a paper's Semantic Scholar ID
result = client.enrich(identifier="10.1038/nature12373")

if result:
    s2_id = result.get('s2_paper_id')
    print(f"Semantic Scholar ID: {s2_id}")

    # Now enrich using the S2 ID
    result_by_id = client.enrich(semantic_scholar_id=s2_id)
    print(f"Retrieved using S2 ID: {result_by_id.get('title')}")

Semantic Scholar ID: a5de30adc5c22bc86e8cfabe7fbd07c052d196a8
Retrieved using S2 ID: Nanometer scale thermometry in a living cell


## Author Enrichment

In [23]:
# First, get an author ID from a paper's authors
result = client.enrich(identifier="10.1038/nature12373")

if result and result.get('authors'):
    first_author = result['authors'][0]
    author_id = first_author.get('author_id')
    print(f"Author: {first_author.get('name')}")
    print(f"Author ID: {author_id}")

    # Enrich author
    if author_id:
        author_info = client.enrich_author(author_id=author_id)
        if author_info:
            print(f"\nAuthor Profile:")
            print(f"Name: {author_info.get('name')}")
            print(f"Paper count: {author_info.get('paper_count')}")
            print(f"Citation count: {author_info.get('citation_count')}")
            print(f"H-index: {author_info.get('h_index')}")
            print(f"Affiliations: {author_info.get('affiliations')}")

Author: None
Author ID: 4301543

Author Profile:
Name: G. Kucsko
Paper count: 23
Citation count: 4077
H-index: 10
Affiliations: []


## Batch Enrichment

Enrich multiple papers at once (up to 500 papers per request).

In [24]:
# List of DOIs to enrich
dois = [
    "10.1038/nature12373",
    "10.1038/nature15758",
    "10.1038/nature16172"
]

# Batch enrich
results = client.enrich_batch(identifiers=dois)

print(f"Successfully enriched {len(results)} papers:\n")
for paper_id, paper_data in results.items():
    print(f"Title: {paper_data.get('title', 'Unknown')[:60]}...")
    print(f"Citations: {paper_data.get('citation_count', 0)}")
    print(f"S2 ID: {paper_id}")
    print("-" * 60)

Successfully enriched 3 papers:

Title: Nanometer scale thermometry in a living cell...
Citations: 1775
S2 ID: a5de30adc5c22bc86e8cfabe7fbd07c052d196a8
------------------------------------------------------------
Title: Migratory neuronal progenitors arise from the neural plate b...
Citations: 144
S2 ID: dce850cff52dda2814084903baf58327c0ad3fa9
------------------------------------------------------------
Title: Ammoniated phyllosilicates with a likely outer Solar System ...
Citations: 311
S2 ID: 0544eb4ab7304c9870fe37eecc0e106b86d406c0
------------------------------------------------------------


## Search Papers

In [25]:
# Search for papers
# Note: For reliable search results, use `bulk=True` to avoid rate limiting issues.
# The default search uses relevance ranking; bulk=True gives faster results without ranking.
query = "machine learning cancer detection"
results = client.search_papers(query=query, limit=5, bulk=True)

print(f"Search results for: '{query}'\n")
for i, paper in enumerate(results, 1):
    print(f"{i}. {paper.get('title', 'Unknown')[:80]}")
    print(f"   Authors: {', '.join([a.get('name', '') for a in paper.get('authors', [])[:3]])}")
    print(f"   Year: {paper.get('year')}, Citations: {paper.get('citation_count')}")
    print(f"   Fields: {paper.get('fields_of_study')}")
    print()

ERROR:pyeuropepmc.enrichment.semantic_scholar:Error searching papers for query 'machine learning cancer detection': [GENERIC002] Paper search failed for query 'machine learning cancer detection': RetryError[<Future at 0x7a3c81bef6a0 state=finished raised ConnectionRefusedError>]


Search results for: 'machine learning cancer detection'



## Paper Recommendations

Get paper recommendations based on a positive example.

In [26]:
# Get recommendations for a single paper
result = client.enrich(identifier="10.1038/nature12373")

if result:
    s2_id = result.get('s2_paper_id')
    print(f"Base paper: {result.get('title')[:60]}...")
    print(f"S2 ID: {s2_id}\n")

    # Get recommendations
    recommendations = client.get_recommendations_for_paper(
        paper_id=s2_id,
        limit=5,
        fields=["title", "authors", "citationCount", "abstract"]
    )

    print(f"Top {len(recommendations)} recommendations:\n")
    for i, paper in enumerate(recommendations, 1):
        print(f"{i}. {paper.get('title', 'Unknown')[:70]}")
        print(f"   Authors: {', '.join([a.get('name', '') for a in paper.get('authors', [])[:3]])}")
        print(f"   Citations: {paper.get('citationCount', 0)}")
        print()

Base paper: Nanometer scale thermometry in a living cell...
S2 ID: a5de30adc5c22bc86e8cfabe7fbd07c052d196a8

Top 5 recommendations:

1. Nanoscale Fluorescence Thermometry: Probes, Recent Advances and Emergi
   Authors: Md. Shakhawat Hossain, Nhat Minh Nguyen, Thi Ngoc Anh Mai
   Citations: 0

2. Peltier-Based Temperature Control Enables High-Bandwidth and Low-Noise
   Authors: Yunxuan Li, Gerardo Guillén, F. Bošković
   Citations: 0

3. Sensing with (One or Many) Upconverting Nanoparticles
   Authors: F. E. Maturi, Erving C Ximendes, A. Benayas
   Citations: 0

4. Multiparametric Quantum Sensing of Liquids Using NV Centres and Tether
   Authors: Johannes Fiedler, Martin Møller Greve, J. Zalieckas
   Citations: 0

5. Fast sweeping for quantum capacitance spectroscopies of two-dimensiona
   Authors: Junhong Chen, Kuan Zhai, Ruiyu Qi
   Citations: 0



## Advanced Recommendations

Get recommendations based on multiple positive and negative examples.

In [27]:
# Get S2 IDs for positive and negative examples
positive_results = client.search_papers(query="deep learning", limit=3)
negative_results = client.search_papers(query="classical statistics", limit=3)

positive_ids = [p.get('s2_paper_id') for p in positive_results if p.get('s2_paper_id')]
negative_ids = [p.get('s2_paper_id') for p in negative_results if p.get('s2_paper_id')]

print(f"Positive examples: {len(positive_ids)} papers")
print(f"Negative examples: {len(negative_ids)} papers\n")

# Get recommendations based on positive and negative examples
recommendations = client.get_recommendations_for_papers(
    positive_paper_ids=positive_ids,
    negative_paper_ids=negative_ids,
    limit=10,
    fields=["title", "authors", "citationCount", "abstract", "venue"]
)

print(f"Recommendations based on {len(positive_ids)} positive and {len(negative_ids)} negative examples:\n")
for i, paper in enumerate(recommendations, 1):
    print(f"{i}. {paper.get('title', 'Unknown')[:70]}")
    print(f"   Venue: {paper.get('venue', 'Unknown')}")
    print(f"   Authors: {', '.join([a.get('name', '') for a in paper.get('authors', [])[:2]])}")
    print(f"   Citations: {paper.get('citationCount', 0)}")
    print()

ERROR:pyeuropepmc.enrichment.semantic_scholar:Error searching papers for query 'deep learning': [GENERIC002] Paper search failed for query 'deep learning': RetryError[<Future at 0x7a3c818fc280 state=finished raised ConnectionRefusedError>]
ERROR:pyeuropepmc.enrichment.semantic_scholar:Error searching papers for query 'classical statistics': [GENERIC002] Paper search failed for query 'classical statistics': RetryError[<Future at 0x7a3c818fcf10 state=finished raised ConnectionRefusedError>]


Positive examples: 0 papers
Negative examples: 0 papers



ValueError: positive_paper_ids must contain at least one paper ID

## Working with Results

Explore all available fields in the enrichment results.

In [ ]:
result = client.enrich(identifier="10.1038/nature12373")

if result:
    print("Available fields:")
    for key, value in result.items():
        if key != 'abstract':  # Skip long abstract for display
            if isinstance(value, (list, dict)):
                print(f"  {key}: {type(value).__name__} with {len(value) if hasattr(value, '__len__') else 'N/A'} items")
            else:
                print(f"  {key}: {value if len(str(value)) < 60 else str(value)[:57] + '...'}")

## Notes

- Rate limiting is automatic (1 second delay between requests by default)
- When API key is missing, rate limiting is 3x more conservative (3x rate_limit_delay)
- Caching is enabled by default when `cache_config` is provided
- API keys provide higher rate limits (100k/month vs 25k for free tier)
- Paper IDs must follow Semantic Scholar's format: alphanumeric, `:`, `.`, `/`, `_`, `-` (max 256 chars)
- The batch API has a 500-paper limit per request
- For best performance, consider setting up a `.env` file with your Semantic Scholar API key